# Main file to compare models on eGFR dataset 

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from kan import KAN, ex_round
from imblearn.over_sampling import SMOTE
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

fold_path = "/home/alecacciatore/rolex/ECML26/GNN4eGFR"
out_path = os.path.join(fold_path, "results_eGFR")
file_path = os.path.join(fold_path, "XY_temp.csv")
file_updated_path = os.path.join(fold_path, "XY_temp_updated.csv")
file_no_temp_path = os.path.join(fold_path, "XY_no_temp_updated.csv")

if not os.path.exists(out_path):
    os.makedirs(out_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Get data

In [2]:
# read the CSV file into a DataFrame
df = pd.read_csv(file_no_temp_path)

# remove icd9 columns
cols_ok = [col for col in df.columns[:-2] if not 'icd9' in col]

# the last column is the target variable and the first one is an indppex
y_classif = df.iloc[:, -1]
y_regress = df.iloc[:, -2]

# X columns
X_no_icd9 = df[cols_ok]
print(X_no_icd9.shape, f"with {X_no_icd9.isnull().sum().sum()} missing values")
X = df.iloc[:, 1:-3] # TODO: include general practitioner features?
print(X.shape, f"with {X.isnull().sum().sum()} missing values")
X = X.fillna(X.mean())
print(X.shape, f"with {X.isnull().sum().sum()} missing values")

# y_classif contains [I, II, IIIa, IIIb, IV, V] labels
# label_mapping = {'I': 0, 'II': 1, 'IIIa': 2, 'IIIb': 3, 'IV': 4, 'V': 5}
label_mapping = {'I': 0, 'II': 1, 'IIIa': 1, 'IIIb': 1, 'IV': 1, 'V': 1}
y_classif = y_classif.map(label_mapping)

# Split data into training and testing sets
X_train, X_test, y_classif_train, y_classif_test = train_test_split(
    X, y_classif, test_size=0.2, random_state=42, stratify=y_classif
)
X_train_reg, X_test_reg, y_regress_train, y_regress_test = train_test_split(
    X, y_regress, test_size=0.2, random_state=42
)

# Create a dataset dictionary for KAN ('test_input', 'test_label', 'train_input', 'train_label')
train_data_classif = {
    'train_input': torch.tensor(X_train.values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_classif_train.values, dtype=torch.long).to(device),
    'test_input': torch.tensor(X_test.values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_classif_test.values, dtype=torch.long).to(device)
}
train_data_regress = {
    'train_input': torch.tensor(X_train_reg.values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_regress_train.values, dtype=torch.float32).to(device),
    'test_input': torch.tensor(X_test_reg.values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_regress_test.values, dtype=torch.float32).to(device)
}

(1833, 498) with 0 missing values
(1833, 539) with 2236 missing values
(1833, 539) with 0 missing values


### Imbalance ratio

In [3]:
class_counts = y_classif_train.value_counts().sort_index()
imbalance_ratio = class_counts.max() / class_counts.min()
print("Class distribution in training set:")
for cls, count in class_counts.items():
    print(f"Class {cls}: {count} samples")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}")

Class distribution in training set:
Class 0: 264 samples
Class 1: 1202 samples
Imbalance Ratio: 4.55


# Define models to be used

In [4]:
# KAN
kan_classifier = KAN(width=[train_data_classif['train_input'].shape[1], 2, 2], grid=3, k=3, device=device, ckpt_path=out_path)

# MLP
mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(4,),      # leggermente più grande per equità
    activation='relu',
    solver='adam',
    alpha=1e-3,                   # regolarizzazione L2
    batch_size='auto',
    learning_rate_init=1e-3,
    max_iter=500,
    random_state=42
)

# SVB (Linear)
from sklearn.svm import SVC

svm_linear = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

# SVB (RBF)
svm_rbf = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

# XGBoost
from xgboost import XGBClassifier

xgb_classifier = XGBClassifier(
    n_estimators=100,        # moderato
    max_depth=3,             # shallow trees
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

checkpoint directory created: /home/alecacciatore/rolex/ECML26/GNN4eGFR/results_eGFR
saving model version 0.0
